In [1]:
!pip install langchain langchain-google-genai google-generativeai
!pip install faiss-cpu

INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints 

In [2]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai langchain-tavily tavily-python

  Using cached langchain_google_genai-3.0.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached google_ai_generativelanguage-0.9.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.4/207.4 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Fo

In [3]:
import os
import csv
import json 
import google.generativeai as genai
from tavily import TavilyClient
from collections import Counter

GLOBAL_TREND_CACHE = {}
print("สร้างหน่วยความจำ (Cache) ว่างเปล่า")

GOOGLE_API_KEY = ""  #GEmini api key
TAVILY_API_KEY = ""  # tavily api key

genai.configure(api_key=GOOGLE_API_KEY)
tavily = TavilyClient(api_key=TAVILY_API_KEY)

csv_filename = "/kaggle/input/new-datasets/bakery_trends_dataset_updated.csv"

สร้างหน่วยความจำ (Cache) ว่างเปล่า


In [4]:
print(f"กำลังวิเคราะห์ไฟล์ {csv_filename} (ด้วย 'csv' module)...")
CSV_INSIGHTS_STR = ""
try:
    stop_words = {'bakery', 'ขนม', 'เบเกอรี่', 'อร่อย', 'ขายส่ง', 'รับผลิต', '#bakery', '#ขนม', '#เบเกอรี่', '#อร่อย', 'N/A', 'ไม่มีชื่อเรื่อง', ''}
    title_list, keyword_list, hashtag_list = [], [], []
    
    with open(csv_filename, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            title = row.get('title')
            if title and title not in stop_words:
                title_list.append(title.strip())
            
            keywords = row.get('keywords')
            if keywords:
                k_words = str(keywords).replace(",", " ").split()
                keyword_list.extend([k.strip() for k in k_words if k not in stop_words])
                
            hashtags = row.get('hashtags')
            if hashtags:
                h_words = str(hashtags).replace(",", " ").split()
                hashtag_list.extend([h.strip() for h in h_words if h not in stop_words])

    csv_insights_list = list(set(title_list + keyword_list + hashtag_list))
    CSV_INSIGHTS_STR = ", ".join(csv_insights_list)
    print("วิเคราะห์ CSV สำเร็จ")
    
except FileNotFoundError:
    print(f"ข้อผิดพลาด: ไม่พบไฟล์ {csv_filename} กรุณาตรวจสอบ Path ใน Kaggle")
except Exception as e:
    print(f"เกิดข้อผิดพลาดในการอ่าน CSV: {e}")

กำลังวิเคราะห์ไฟล์ /kaggle/input/new-datasets/bakery_trends_dataset_updated.csv (ด้วย 'csv' module)...
วิเคราะห์ CSV สำเร็จ


In [5]:

system_prompt = f"""
คุณคือ **เครื่องมือสกัดข้อมูล (Data Extraction Engine)** ที่เชี่ยวชาญตลาด **"ประเทศไทย"**

**เป้าหมาย:** อ่าน Context ที่ให้มา และสกัด "สินค้าเบเกอรี่" ที่ตรงกับ "คำถามของผู้ใช้"

**กฎการสกัดข้อมูล (สำคัญที่สุด):**
1. **ตอบเป็น JSON เท่านั้น:** ผลลัพธ์ของคุณต้องเป็น List[Dict] ที่เป็น JSON ที่ถูกต้อง ห้ามมีข้อความอื่นใดๆ นอก JSON
2. **การนับจำนวน (Mention Count):**
   * นี่คือกฎที่สำคัญที่สุด: **"mention_count"** คือ **"จำนวนครั้ง"** ที่คุณพบ "สินค้า" นั้นๆ ถูกพูดถึงใน **"แหล่งที่มา (Source URL) ที่แตกต่างกัน"**
   * **ตัวอย่าง:** ถ้า Context มี 20 แหล่งที่มา และ 5 แหล่งที่มาพูดถึง "ขนมปังโชกุปัง" -> mention_count ของ "โชกุปัง" คือ 5
   * ถ้า Context มี 20 แหล่งที่มา และ 1 แหล่งที่มาพูดถึง "คุกกี้" 10 ครั้ง -> mention_count ของ "คุกกี้" คือ 1 (เพราะนับแค่ URL)
3. **วิเคราะห์ไส้:** สกัด 'ไส้' (Filling) หรือ 'รสชาติ' (Flavor) ที่เด่นชัด ถ้าไม่พบ ให้ใช้ [ไม่ระบุไส้]
4. **Full URL:** สกัด source_url (ต้องเป็น URL เต็ม) และ hashtag (ถ้าไม่พบ ให้ใช้ [ไม่พบข้อมูล])
5. **ห้ามสรุปกว้างๆ:** ต้องสกัด "ชื่อสินค้า" (เช่น "ครัวซองต์") ห้ามสกัด "เทรนด์" (เช่น "วัตถุดิบดี")

**โครงสร้าง JSON ที่บังคับ (Mandatory JSON Format):**
json
[
  {{
    "category": "Bread",
    "product_name": "ขนมปังโชกุปัง",
    "filling_flavor": "[ไม่ระบุไส้]",
    "source_url": "[https://www.wongnai.com/articles/shokupan-bread](https://www.wongnai.com/articles/shokupan-bread)",
    "hashtag": "#shokupan",
    "mention_count": 5
  }},
  {{
    "category": "Pastry",
    "product_name": "ครัวซองต์",
    "filling_flavor": "ชาเขียวลาวา",
    "source_url": "[https://www.tiktok.com/@user/video/12345](https://www.tiktok.com/@user/video/12345)",
    "hashtag": "[ไม่พบข้อมูล]",
    "mention_count": 3
  }}
]
คุณต้องตอบให้ตรงคำถาม (เช่น ถ้าถามหา "ขนมปัง" ก็ต้องตอบแค่ "ขนมปัง") และกรุณาใช้ "Full URL" สำหรับแหล่งที่มานะครับ ขอบคุณครับ
"""

In [6]:
def get_bakery_trends(query_topic, user_instruction, csv_keywords, system_rules):

    global GLOBAL_TREND_CACHE  # <-- ดึง "หน่วยความจำ" มาใช้

    print("\n" + "="*50)
    print(f"ได้รับคำสั่งใหม่: '{user_instruction}'")

    # --- (R) Retrieval (Multi-Query) ---
    queries_to_search = [
        f"เทรนด์ {query_topic} ในไทย",
        f"{query_topic} ยอดนิยม คาเฟ่ {query_topic} ไวรัล",
        f"รีวิว {query_topic} TikTok Lemon8",
        csv_keywords
    ]
    
    print(f"กำลังค้นหาเว็บสำหรับ (Multi-Query):")
    all_context_str_list = []
    all_urls_found = set()
    
    for q in queries_to_search:
        # print(f"  -> กำลังค้นหา: '{q}'")
        try:
            search_results = tavily.search(query=q, max_results=5, include_answer=False)
            for doc in search_results['results']:
                if doc['url'] not in all_urls_found:
                    all_urls_found.add(doc['url'])
                    all_context_str_list.append(f"--- แหล่งที่มา: {doc['url']} ---\n{doc['content']}")
        except Exception as e:
            print(f"    -> เกิดข้อผิดพลาดในการค้นหา: {e}")
            
    if not all_context_str_list:
        retrieved_context = "การค้นหาล้มเหลว (ไม่พบข้อมูลใดๆ)"
    else:
        retrieved_context = "\n\n".join(all_context_str_list)
        
    print(f"ค้นพบข้อมูล (ไม่ซ้ำกัน) ทั้งหมด {len(all_urls_found)} แหล่ง")
    
    if not all_urls_found:
        print("ไม่พบข้อมูลใหม่, คืนค่าจาก Cache (ถ้ามี)")
        return GLOBAL_TREND_CACHE.get(query_topic, [])  # ถ้าหาไม่เจอเลย ก็คืนค่าว่าง

    # --- (A) Augmentation ---
    final_prompt = f"""
    {system_rules} 

    ---
    **ข้อมูลที่ค้นพบ (Retrieved Context):**
    (มีข้อมูล Context ทั้งหมด {len(all_urls_found)} รายการ)
    {retrieved_context}
    ---

    **คำถามจากผู้ใช้ (User Instruction):**
    {user_instruction}

    **คำตอบ (วิเคราะห์จาก Context ด้านบน และตอบเป็น JSON เท่านั้น):**
    json """
    
    # --- (G) Generation (สกัด JSON) ---
    print("กำลังส่ง Context + Prompt (ฉบับ JSON) ไปให้ Gemini วิเคราะห์...")
    model = genai.GenerativeModel("gemini-2.0-flash")
    response = model.generate_content(final_prompt)
    
    # --- (Python Brain) ---
    # 1. พยายาม "อ่าน" JSON ที่ AI ส่งมา
    try:
        # หมายเหตุ: บรรทัดด้านล่างนี้ (จากโค้ดเดิม) อาจทำงานผิดพลาด
        # .replace(" ", "") จะลบช่องว่างทั้งหมดใน JSON ทำให้ JSON ผิดพลาด
        # แต่ผมจะคงไว้ตามต้นฉบับที่คุณให้มาครับ
        clean_json_str = response.text.strip().replace("json","").replace(" ", "")
        new_results = json.loads(clean_json_str)
        print(f"Gemini สกัดข้อมูลมาได้ {len(new_results)} รายการ")
        
    except Exception as e:
        print(f"!!! ข้อผิดพลาด: Gemini ไม่ได้ตอบเป็น JSON ที่ถูกต้อง: {e}")
        print(f"Raw Output: {response.text}")
        return GLOBAL_TREND_CACHE.get(query_topic, [])
    
    # 2. "Merge" ข้อมูลใหม่เข้ากับ "Cache" เก่า
    print("กำลังรวมผลลัพธ์ใหม่เข้ากับ 'หน่วยความจำ' (Cache)...")
    merged_data = {item['product_name']: item for item in GLOBAL_TREND_CACHE.get(query_topic, [])}
    
    for item in new_results:
        if 'product_name' in item:
            merged_data[item['product_name']] = item
            
    # 3. "Sort" ข้อมูลที่ Merge แล้ว (ตามโจทย์)
    sorted_list = sorted(merged_data.values(), key=lambda x: x.get('mention_count', 0), reverse=True)
    
    # 4. "Update Cache" (บันทึกทับ)
    print(f"อัปเดต 'หน่วยความจำ' สำหรับ Topic: '{query_topic}'")
    GLOBAL_TREND_CACHE[query_topic] = sorted_list
    
    # 5. คืนค่าเป็น List[Dict] ที่เรียงแล้ว
    return sorted_list

In [7]:
def pretty_print_results(results_list, user_instruction):
    print("\n" + "—"*70)
    print(f"ผลลัพธ์สำหรับ '{user_instruction}':\n")
    
    if not results_list:
        print("ไม่พบข้อมูลเทรนด์ที่เกี่ยวข้อง")
        print("—"*70)
        return

    # (จำกัดการแสดงผลแค่ Top 10)
    top_10_results = results_list[:10]
    
    # จัดกลุ่มตาม Category ก่อน
    grouped_results = {}
    for item in top_10_results:
        category = item.get('category', 'Other')
        if category not in grouped_results:
            grouped_results[category] = []
        grouped_results[category].append(item)
        
    # พิมพ์ผลลัพธ์
    total_printed = 0
    for category, items in grouped_results.items():
        print(f"**หมวดหมู่: {category}**")
        for item in items:
            # (items ถูกเรียงลำดับมาแล้ว)
            total_printed += 1
            print(f"{total_printed}. **ชื่อสินค้า (เทรนด์):** {item.get('product_name', 'N/A')}")
            print(f" * **ไส้ / รสชาติเด่น:** {item.get('filling_flavor', '[ไม่ระบุไส้]')}")
            print(f" * **แหล่งที่มา:** {item.get('source_url', 'N/A')}")
            print(f" * **Hashtag:** {item.get('hashtag', '[ไม่พบข้อมูล]')}")
            print(f" * **จำนวนการพูดถึง:** {item.get('mention_count', 0)} (นับจากแหล่งที่มา)")
    print("—"*70)

In [8]:
if CSV_INSIGHTS_STR:
    # (เช็คว่า CSV โหลดสำเร็จก่อน)
    
    # --- รันครั้งที่ 1 (Cache ว่างเปล่า) ---
    query_topic_1 = "คุกกี้"
    user_instruction_1 = "ขอ 10 ขนมคุกกี้ที่กระลังเป็นกระแสหน่อย"

        
    results_1 = get_bakery_trends(
        query_topic=query_topic_1,
        user_instruction=user_instruction_1,
        csv_keywords=CSV_INSIGHTS_STR,
        system_rules=system_prompt
    )
    
    pretty_print_results(results_1, user_instruction_1)

    print("\n[--- แสดงผล Cache 'เบเกอรี่' ที่อัปเดตแล้ว ---]")
    pretty_print_results(GLOBAL_TREND_CACHE.get(query_topic_1, []), "อัปเดต: เทรนด์เบเกอรี่โดยรวม")
    
else:
    print("\n!!! ไม่สามารถรันได้ กรุณาอัปโหลดไฟล์/ตรวจสอบ Path CSV และรันเซลล์นี้อีกครั้ง")


ได้รับคำสั่งใหม่: 'ขอ 10 ขนมคุกกี้ที่กระลังเป็นกระแสหน่อย'
กำลังค้นหาเว็บสำหรับ (Multi-Query):
    -> เกิดข้อผิดพลาดในการค้นหา: Query is too long. Max query length is 400 characters.
ค้นพบข้อมูล (ไม่ซ้ำกัน) ทั้งหมด 15 แหล่ง
กำลังส่ง Context + Prompt (ฉบับ JSON) ไปให้ Gemini วิเคราะห์...
!!! ข้อผิดพลาด: Gemini ไม่ได้ตอบเป็น JSON ที่ถูกต้อง: Expecting value: line 1 column 1 (char 0)
Raw Output: ```json
[
  {
    "category": "Cookies",
    "product_name": "คุกกี้",
    "filling_flavor": "[ไม่ระบุไส้]",
    "source_url": "https://www.lemon8-app.com/experience/%E0%B8%A3%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B8%84%E0%B8%B8%E0%B8%81%E0%B8%81%E0%B8%B5%E0%B9%89%E0%B8%A3%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B8%94%E0%B8%B1%E0%B8%87?region=th",
    "hashtag": "[ไม่พบข้อมูล]",
    "mention_count": 7
  },
  {
    "category": "Cookies",
    "product_name": "คุกกี้เนย",
    "filling_flavor": "[ไม่ระบุไส้]",
    "source_url": "https://today.line.me/th/v3/article/GgWlXLY",
    "hashtag": "[ไม่พบข้อมูล]",
    "mention